# Map of the average number of suicides by autonomous community in Spain

**What this notebook does.** Computes the average yearly number of suicides of each Spanish autonomous community (2001–2024) and draws it on a choropleth map.

**Inputs**
- `data/raw/datos_ine_suicidio_ccaa_desde2001.csv`: suicide deaths by autonomous community and year (INE, not in GitHub; see the README to download it).
- Internet access: boundaries of the autonomous communities (Ministerio de Transportes, `mapas.fomento.gob.es`).

**Outputs.** None: the map is shown in the notebook.

**How to run.** Open the notebook from the `notebooks/` folder and use *Run All*.

In [ ]:
from pathlib import Path

# Repository root: the first folder, going up from this notebook, that contains "data/"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
RAW = ROOT / "data" / "raw"              # original downloads (not uploaded to GitHub)
INTERIM = ROOT / "data" / "interim"      # large intermediate files (not uploaded to GitHub)
PROCESSED = ROOT / "data" / "processed"  # clean tables used in the analysis

In [4]:

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import unicodedata
import re

# ---------------------------------------
# 1. NORMALIZAR LOS NOMBRES DEL INE
# ---------------------------------------

nombres_ine = {
    "Asturias, Principado de": "Principado de Asturias",
    "Balears, Illes": "Illes Balears",
    "Castilla - La Mancha": "Castilla-La Mancha",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Madrid, Comunidad de": "Comunidad de Madrid",
    "Murcia, Región de": "Región de Murcia",
    "Navarra, Comunidad Foral de": "Comunidad Foral de Navarra",
    "Rioja, La": "La Rioja"
}

# ---------------------------------------
# 2. CARGAR LOS DATOS DEL INE
# ---------------------------------------

df = pd.read_csv(
    RAW / "datos_ine_suicidio_ccaa_desde2001.csv",
    encoding="latin1",
    sep=None,
    engine="python"
)

# Seleccionar todos los años
df_year = df.copy()

# Si quieres seleccionar un año concreto:
# year = 2012
# df_year = df[df["Periodo"] == year].copy()

# ---------------------------------------
# 3. PREPARAR LOS DATOS
# ---------------------------------------

df_year["CCAA"] = (
    df_year["Comunidades y Ciudades Autónomas"]
    .str.replace(r"^\d+\s+", "", regex=True)
)

# Eliminar registros que no corresponden a CCAA
df_year = df_year[df_year["CCAA"] != "Extranjero"].copy()

# Adaptar los nombres
df_year["CCAA"] = df_year["CCAA"].replace(nombres_ine)

# Calcular el promedio por comunidad autónoma
df_year = (
    df_year.groupby("CCAA", as_index=False)["Total"]
    .mean()
)

df_year["Total"] = df_year["Total"].round(2)

# ---------------------------------------
# 4. NORMALIZAR LOS NOMBRES
# ---------------------------------------

def normalizar(nombre):
    nombre = unicodedata.normalize("NFKD", str(nombre))
    nombre = "".join(
        c for c in nombre
        if not unicodedata.combining(c)
    )
    nombre = nombre.lower().strip()
    nombre = re.sub(r"[-_,]", " ", nombre)
    return " ".join(nombre.split())

df_year["join_name"] = df_year["CCAA"].apply(normalizar)

# ---------------------------------------
# 5. CARGAR LOS LÍMITES GEOGRÁFICOS
# ---------------------------------------

url = (
    "https://mapas.fomento.gob.es/arcgis/rest/services/"
    "SIU/ENTIDADES_TERRITORIALES_EGRN/MapServer/1/query"
    "?where=1%3D1&outFields=*&outSR=4326&f=geojson"
)

gdf = gpd.read_file(url)

print(gdf.columns)

# ---------------------------------------
# 6. PREPARAR LA UNIÓN DE LOS DATOS
# ---------------------------------------

gdf["join_name"] = gdf["NAMEUNIT"].apply(normalizar)

mapa = gdf.merge(
    df_year[["join_name", "Total"]],
    on="join_name",
    how="left"
)

# Comprobar comunidades sin datos
print(mapa[["NAMEUNIT", "Total"]])

# ---------------------------------------
# 7. REPRESENTAR EL MAPA
# ---------------------------------------

fig, ax = plt.subplots(figsize=(12, 10))

mapa.plot(
    column="Total",
    cmap="Reds",
    linewidth=0.8,
    edgecolor="black",
    legend=True,
    legend_kwds={
        "label": "Número medio de suicidios",
        "orientation": "horizontal",
        "shrink": 0.7
    },
    missing_kwds={
        "color": "lightgrey",
        "label": "Sin datos"
    },
    ax=ax
)

ax.set_title(
    "Promedio de suicidios por comunidad autónoma en España",
    fontsize=16,
    pad=20
)

ax.axis("off")

plt.tight_layout()
plt.show()


IndentationError: unexpected indent (794285217.py, line 6)